# 05 — Inventory Risk Intelligence & Decisioning

**Deliverable D4 Companion Notebook**

This notebook evaluates the 4-quadrant inventory risk decisioning engine, reconciles chain vs store granularity, and verifies classification performance against the ground-truth anomaly answer key (`sku_inventory_flags.csv`).

### Core Governance:
- **Evaluation Cutoff Duality:** Model evaluated at Week 41, 2025 cutoff (before the Oct 17 – Dec 31 anomaly window) to prove pre-stockout detection.
- **Rule 3 Compliance:** `sku_inventory_flags.csv` is read strictly by the evaluation harness; it never enters training or features.

In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

from foresight.config import load_settings

settings = load_settings()
artifacts = settings.artifacts_dir
risk_scores = pd.read_parquet(artifacts / 'risk' / 'risk_scores.parquet')
eval_json = json.loads((artifacts / 'metrics' / 'flag_evaluation.json').read_text(encoding='utf-8'))
print(f"Loaded risk scores for {len(risk_scores):,} inventory SKUs.")

Loaded risk scores for 4,495 inventory SKUs.


## 1. Production 4-Quadrant Distribution

Every SKU is classified into one of four operational quadrants:
- **REORDER NOW:** High stockout risk, immediate PO required.
- **HEALTHY:** Balanced stock, optimal forward cover (4-8 weeks).
- **MARKDOWN / CLEAR:** High overstock, promotional markdown candidate.
- **WATCH / VOLATILE:** Volatile demand, restock review required.

In [2]:
prod_summary = eval_json['production']
display(pd.DataFrame(list(prod_summary['quadrant_counts'].items()), columns=['Quadrant', 'SKU Count']))

print(f"\nTotal Working Capital Locked in Excess Stock: INR {prod_summary['locked_capital']:,.0f}")
print(f"Total Sales at Imminent Risk of Stockout:     INR {prod_summary['rupees_at_risk']:,.0f}")

,Quadrant,SKU Count
0,healthy,1149
1,markdown_clear,3021
2,reorder_now,325
3,watch_volatile,0



Total Working Capital Locked in Excess Stock: INR 1,534,592,370
Total Sales at Imminent Risk of Stockout:     INR 32,785,603


## 2. Validation Against Ground-Truth Anomaly Key

We evaluate detection metrics (Recall, Precision, F1) against `sku_inventory_flags.csv`:
- Target: Detect $\ge 85\%$ of imminent stockouts.

In [3]:
eval_data = eval_json['evaluation']

eval_rows = [
    {
        'Anomaly Type': 'STOCKOUT_RISK (SKU)',
        'Actual Key Anomalies': eval_data['stockout_risk']['n_actual'],
        'Predicted Anomalies': eval_data['stockout_risk']['n_predicted'],
        'True Positives': eval_data['stockout_risk']['tp'],
        'False Positives': eval_data['stockout_risk']['fp'],
        'False Negatives': eval_data['stockout_risk']['fn'],
        'Recall': f"{eval_data['stockout_risk']['recall']:.1%}",
        'Precision': f"{eval_data['stockout_risk']['precision']:.1%}",
        'F1-Score': f"{eval_data['stockout_risk']['f1']:.3f}"
    },
    {
        'Anomaly Type': 'SLOW_MOVER (SKU)',
        'Actual Key Anomalies': eval_data['slow_mover']['n_actual'],
        'Predicted Anomalies': eval_data['slow_mover']['n_predicted'],
        'True Positives': eval_data['slow_mover']['tp'],
        'False Positives': eval_data['slow_mover']['fp'],
        'False Negatives': eval_data['slow_mover']['fn'],
        'Recall': f"{eval_data['slow_mover']['recall']:.1%}",
        'Precision': f"{eval_data['slow_mover']['precision']:.1%}",
        'F1-Score': f"{eval_data['slow_mover']['f1']:.3f}"
    },
    {
        'Anomaly Type': 'Store-SKU Outage Pairs',
        'Actual Key Anomalies': eval_data['store_pairs']['n_actual'],
        'Predicted Anomalies': eval_data['store_pairs']['n_predicted'],
        'True Positives': eval_data['store_pairs']['tp'],
        'False Positives': eval_data['store_pairs']['fp'],
        'False Negatives': eval_data['store_pairs']['fn'],
        'Recall': f"{eval_data['store_pairs']['recall']:.1%}",
        'Precision': f"{eval_data['store_pairs']['precision']:.1%}",
        'F1-Score': f"{eval_data['store_pairs']['f1']:.3f}"
    }
]

display(pd.DataFrame(eval_rows))
print(f"\nStockout Recall Target (85.0%): {'MET' if eval_data['meets_stockout_recall_target'] else 'NOT MET'} ({eval_data['stockout_risk']['recall']:.1%})")

,Anomaly Type,Actual Key Anomalies,Predicted Anomalies,True Positives,False Positives,False Negatives,Recall,Precision,F1-Score
0,STOCKOUT_RISK (SKU),200,382,173,209,27,86.5%,45.3%,0.595
1,SLOW_MOVER (SKU),400,517,400,117,0,100.0%,77.4%,0.872
2,Store-SKU Outage Pairs,2703,4771,2703,2068,0,100.0%,56.7%,0.723



Stockout Recall Target (85.0%): MET (86.5%)


## 3. Interactive Decision Grid Scatter Plot

We plot Stockout Risk vs. Weeks of Supply (Forward Cover), sized by Rupee Impact and colored by quadrant.

In [4]:
sample_plot = risk_scores.sample(n=min(1000, len(risk_scores)), random_state=42).copy()
sample_plot['rupee_impact'] = sample_plot['rupees_at_risk'] + sample_plot['locked_capital']

fig = px.scatter(
    sample_plot,
    x='wos',
    y='stockout_risk',
    color='quadrant',
    size='rupee_impact',
    hover_data=['sku_id', 'stock_on_hand', 'forward_weekly_mean', 'lead_time_days'],
    labels={'wos': 'Weeks of Supply (Forward Cover)', 'stockout_risk': 'Stockout Risk Probability'},
    title='Zidio 4-Quadrant Decision Grid (1,000 SKU Representative Sample)',
    template='plotly_white'
)
fig.show()

## 4. Deliverable D4 Sign-Off

- **Stockout Recall:** 86.5% (Target $\ge 85\%$ achieved)
- **Store Outage Coverage:** 100% of ground-truth store outages captured
- **Decision Engine Outputs:** Saved to `artifacts/risk/risk_scores.parquet` and `artifacts/risk/store_risk.parquet`
- **Audit memo:** `reports/risk_evaluation.md`.